In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import Row


dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

catalog = f"{env}_lakehouse"
#api_key = dbutils.secrets.get(scope = "lakehouse", key = "alpha_vantage_api_key")



In [0]:
dbutils.widgets.text("api_key", "")
api_key = dbutils.widgets.get("api_key")

BASE_URL = "https://www.alphavantage.co/query"


In [0]:
symbols = ["AAPL", "MSFT", "GOOGL"]

In [0]:
rows = []

for symbol in symbols:
    try:
        params = {
            "function": "TIME_SERIES_DAILY",
            "symbol": symbol,
            "apikey": api_key
        }

        response = requests.get(BASE_URL, params=params, timeout=10)

        ingestion_time = datetime.utcnow()

        if response.status_code == 200:
            rows.append(
                Row(
                    raw_payload=response.text,
                    symbol=symbol,
                    ingestion_ts=ingestion_time,
                    source="alpha_vantage",
                    api_status="SUCCESS"
                )
            )
        else:
            rows.append(
                Row(
                    raw_payload=response.text,
                    symbol=symbol,
                    ingestion_ts=ingestion_time,
                    source="alpha_vantage",
                    api_status="HTTP_ERROR"
                )
            )

    except Exception as e:
        rows.append(
            Row(
                raw_payload=str(e),
                symbol=symbol,
                ingestion_ts=datetime.utcnow(),
                source="alpha_vantage",
                api_status="EXCEPTION"
            )
        )


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

bronze_schema = StructType([
    StructField("raw_payload", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("ingestion_ts", TimestampType(), True),
    StructField("source", StringType(), True),
    StructField("api_status", StringType(), True)
])


In [0]:
bronze_df = spark.createDataFrame(rows, schema=bronze_schema)


In [0]:
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog}.bronze.stock_prices_raw")
